# RealNVP — coupling layers, Two Moons i likelihood w PyTorch

> **Cel notebooka:** od podstaw zaimplementować RealNVP, wyprowadzić coupling affine, trenować model na Two Moons i analizować próbki oraz gęstość. Kod nie korzysta z gotowej biblioteki flow.

RealNVP (Real-valued Non-Volume Preserving) rozwiązuje kluczowy problem: jak uzyskać nieliniową transformację, której odwrotność i log-wyznacznik są tanie. Odpowiedzią jest **affine coupling layer** — część wymiarów pozostaje bez zmian i przewiduje parametry transformacji pozostałych wymiarów.

**Plan:** Two Moons → algebra coupling → implementacja PyTorch → trening NLL → wizualizacja gęstości i próbek → analiza i eksperymenty.


## 1. Dane Two Moons

Dwa półksiężyce są niskowymiarowym, nieliniowym rozkładem z dwoma połączonymi, zakrzywionymi obszarami. Idealne do wizualizacji: widać zarówno, gdzie model umieszcza masę, jak i jak wygląda odwzorowanie z gaussowskiej bazy.

Tworzymy dane syntetycznie, więc notebook działa bez pobierania zbioru. Podział train/validation zapobiega ocenianiu dopasowania tylko na przykładach treningowych.


In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

X, _ = make_moons(n_samples=12000, noise=0.06, random_state=SEED)
X = torch.tensor(X, dtype=torch.float32)
# Standaryzacja pomaga optymalizatorowi; gęstość raportujemy w tej przestrzeni.
mean, std = X[:9000].mean(0), X[:9000].std(0)
X = (X - mean) / std
perm = torch.randperm(len(X)); train = X[perm[:9000]].to(device); valid = X[perm[9000:]].to(device)
plt.figure(figsize=(5, 4)); plt.scatter(train[:2000,0].cpu(), train[:2000,1].cpu(), s=4, alpha=.35); plt.axis('equal'); plt.title('Two Moons — dane treningowe'); plt.show()

## 2. Affine coupling: wyprowadzenie

Podzielmy wektor $x=(x_a,x_b)$, gdzie $x_a$ jest warunkującą częścią. Sieć $s,t$ bierze $x_a$ i zwraca wektory tej samej długości co $x_b$:
$$y_a=x_a,\qquad y_b=x_b\odot\exp(s(x_a))+t(x_a).$$

Odwrotność jest natychmiastowa, bo $y_a=x_a$ dostarcza te same parametry:
$$x_a=y_a,\qquad x_b=(y_b-t(y_a))\odot\exp(-s(y_a)).$$

Jakobian ma blokową postać trójkątną:
$$J=\begin{bmatrix}I&0\\ *&\operatorname{diag}(\exp(s))\end{bmatrix},\quad \log|\det J|=\sum_j s_j(x_a).$$

To kluczowy trik: sieci $s,t$ mogą być nieliniowe i głębokie, ale log-wyznacznik nie wymaga autodiff po każdym elemencie. Aby każdy wymiar czasem pełnił rolę $a$, przeplatamy maski/permutacje między warstwami. W 2D maska naprzemiennie zachowuje x albo y.

Skalę często ogranicza się przez $s=c\tanh(\hat s)$, bo niekontrolowane $\exp(s)$ może wybuchać lub zanikać. Stała $c$ reguluje maksymalną log-skale.


In [ ]:
class CouplingNet(nn.Module):
    """Mały MLP przewidujący przesunięcie i surową log-skalę."""
    def __init__(self, in_dim, out_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.Tanh(),
                                  nn.Linear(hidden, hidden), nn.Tanh(),
                                  nn.Linear(hidden, 2*out_dim))
        # Start blisko identyczności: stabilniejszy początek optymalizacji.
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
        self.out_dim = out_dim
    def forward(self, x):
        shift, raw_scale = self.net(x).chunk(2, dim=-1)
        log_scale = 1.5 * torch.tanh(raw_scale)
        return shift, log_scale

class AffineCoupling(nn.Module):
    def __init__(self, dim, mask):
        super().__init__()
        self.register_buffer('mask', torch.tensor(mask, dtype=torch.float32))
        self.net = CouplingNet(dim, dim)
    def forward(self, x):
        xa = x * self.mask
        shift, log_scale = self.net(xa)
        active = 1.0 - self.mask
        y = xa + active * (x * torch.exp(log_scale) + shift)
        logdet = (log_scale * active).sum(dim=-1)
        return y, logdet
    def inverse(self, y):
        ya = y * self.mask
        shift, log_scale = self.net(ya)
        active = 1.0 - self.mask
        x = ya + active * ((y - shift) * torch.exp(-log_scale))
        inverse_logdet = -(log_scale * active).sum(dim=-1)
        return x, inverse_logdet

layer = AffineCoupling(2, [1, 0]).to(device)
test_x = torch.randn(5, 2, device=device)
test_y, ld = layer(test_x); recovered, ild = layer.inverse(test_y)
print('Maksymalny błąd odwrotności:', (test_x-recovered).abs().max().item())
print('Logdet + odwrotny logdet ~ 0:', (ld+ild).abs().max().item())

## 3. Cały przepływ i gęstość

Dla kompozycji warstw przepuszczamy dane przez warstwy, sumując log-determinanty. Log-gęstość w bazie to suma log-gęstości niezależnych standardowych Gaussianów. Próbkowanie idzie w drugą stronę: losujemy bazę, a następnie stosujemy odwrotności warstw w odwróconej kolejności.

Kolejność jest ważna: jeśli $z=f_K(\cdots f_1(x))$, inverse zaczyna od $f_K^{-1}$. W naszym API forward mapuje dane do bazy (ułatwia likelihood), inverse mapuje bazę do danych.


In [ ]:
class RealNVP(nn.Module):
    def __init__(self, dim=2, n_layers=8):
        super().__init__()
        masks = [[1., 0.] if i % 2 == 0 else [0., 1.] for i in range(n_layers)]
        self.layers = nn.ModuleList([AffineCoupling(dim, m) for m in masks])
        self.dim = dim
    def to_base(self, x):
        logdet = torch.zeros(x.shape[0], device=x.device)
        z = x
        for layer in self.layers:
            z, ld = layer(z); logdet += ld
        return z, logdet
    def log_prob(self, x):
        z, logdet = self.to_base(x)
        log_pz = -0.5 * (z.square() + math.log(2*math.pi)).sum(-1)
        return log_pz + logdet  # forward to base: + log|det dz/dx|
    @torch.no_grad()
    def sample(self, n):
        z = torch.randn(n, self.dim, device=next(self.parameters()).device)
        for layer in reversed(self.layers): z, _ = layer.inverse(z)
        return z

flow = RealNVP(n_layers=8).to(device)
print('Prawdopodobieństwa testowe:', flow.log_prob(test_x))


## 4. Trening maximum likelihood

Minimalizujemy $-\mathbb E_{x\sim p_{data}}[\log p_\theta(x)]$. W przeciwieństwie do GAN-a nie ma dyskryminatora ani heurystycznej funkcji celu: gęstość wynika bezpośrednio z bazy i jakobianu. Walidacyjne NLL liczymy na niewidzianych przykładach.

Konfigurację można zmniejszyć, jeśli środowisko jest wolne. Dla 2D trening powinien być krótki; losowość minibatchy miesza kolejność obserwacji.


In [ ]:
epochs, batch_size = 80, 512
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-3, weight_decay=1e-6)
train_history, valid_history = [], []
for epoch in range(epochs):
    flow.train(); order = torch.randperm(len(train), device=device); losses=[]
    for idx in order.split(batch_size):
        loss = -flow.log_prob(train[idx]).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(flow.parameters(), 10.0)
        optimizer.step(); losses.append(loss.detach())
    flow.eval()
    with torch.no_grad(): val_nll = -flow.log_prob(valid).mean().item()
    train_nll = torch.stack(losses).mean().item()
    train_history.append(train_nll); valid_history.append(val_nll)
    if (epoch+1) % 10 == 0: print(f'epoch {epoch+1:3d} | train NLL {train_nll:7.3f} | valid NLL {val_nll:7.3f}')

plt.figure(figsize=(7,3)); plt.plot(train_history,label='train'); plt.plot(valid_history,label='validation'); plt.xlabel('Epoka'); plt.ylabel('NLL [naty log]'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 5. Próbki i mapa gęstości

Likelihood i próbki odpowiadają na różne pytania. Kontury gęstości pokażą, gdzie model przypisuje masę; losowe próbki pokażą, czy cały obszar dwóch półksiężyców jest pokryty bez luk lub artefaktów. Gładka gęstość na wykresie nie gwarantuje idealnej jakości modelu, ale dla tego syntetycznego problemu stanowi czytelną diagnostykę.


In [ ]:
flow.eval()
with torch.no_grad():
    samples = flow.sample(5000).cpu()
    lo = train.min(0).values.cpu() - 1.0; hi = train.max(0).values.cpu() + 1.0
    gx, gy = torch.meshgrid(torch.linspace(lo[0],hi[0],180), torch.linspace(lo[1],hi[1],180), indexing='xy')
    points = torch.stack([gx.flatten(),gy.flatten()], dim=-1).to(device)
    logp = flow.log_prob(points).reshape(gx.shape).cpu()

fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(valid[:2500,0].cpu(), valid[:2500,1].cpu(),s=4,alpha=.35); ax[0].set_title('Rzeczywiste dane walidacyjne')
ax[1].scatter(samples[:,0],samples[:,1],s=4,alpha=.3); ax[1].set_title('Próbki RealNVP')
for a in ax: a.set_aspect('equal'); a.grid(alpha=.15)
plt.show()
plt.figure(figsize=(6,5)); plt.contourf(gx,gy,logp,levels=35,cmap='viridis'); plt.colorbar(label='log p(x)'); plt.scatter(valid[:1500,0].cpu(),valid[:1500,1].cpu(),s=2,c='white',alpha=.35); plt.title('Log-gęstość modelu i dane walidacyjne'); plt.axis('equal'); plt.show()

## 6. Analiza, błędy i eksperymenty

**Typowe błędy implementacyjne**
- Zły znak: przy mapowaniu danych $x\to z$ dodajemy $\log|\det dz/dx|$ do log-gęstości bazy.
- Pominięcie sumy po wymiarze w log-gęstości bazowej lub po transformowanych wymiarach w logdet.
- Odwracanie warstw w tej samej kolejności zamiast w odwrotnej.
- Brak zmiany masek: jeden wymiar nigdy nie jest transformowany.
- Zbyt duża skala: `exp(log_scale)` może prowadzić do przepełnień; ograniczenie przez `tanh` pomaga.
- Użycie testu zamiast validation podczas strojenia.

**Interpretacja NLL:** wartość jest w nat-ach na przykład (2D). Porównuj ją tylko przy tej samej reprezentacji i wymiarze. Standaryzacja danych zmienia log-gęstość o stałą Jacobianu tej transformacji; tutaj wykres i NLL odnoszą się do standaryzowanej przestrzeni.

**Eksperymenty:** zmień liczbę warstw (2, 4, 8, 12), szerokość MLP, limit log-skali, szum Two Moons i learning rate. Zapisuj train/validation NLL oraz próbki. Sprawdź odwracalność dla kilku batchy po treningu.

**Ograniczenia RealNVP:** coupling warstwy zostawiają część wejścia bez zmian na danym kroku, więc potrzeba wielu warstw i permutacji; zwykłe affine coupling ogranicza lokalną rodzinę transformacji. Dla obrazów RealNVP wymaga wieloskalowej architektury, squeeze i wielkich sieci, więc niniejszy przykład celowo koncentruje się na mechanice, nie jakości obrazu.

## Podsumowanie

Affine coupling daje odwracalną nieliniową transformację o prostym odwrotnym wzorze i logdet równym sumie log-skali. Warstwy naprzemienne zapewniają, że każdy wymiar zostaje przekształcony. Maximum likelihood uczy flow bezpośrednio dopasowywać gęstość, a z bazy możemy łatwo generować próbki.
